# **Basic Audio Project**

## **Audio Data**

In [1]:
# 라이브러리 불러오기
import os
import sys
import math
import random
import shutil
import warnings

# 경고 무시
if not sys.warnoptions:
    warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import IPython.display as ipd

import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision

from tqdm import tqdm

from sklearn.model_selection import train_test_split, KFold

In [2]:
# 하이퍼파라미터
args = {
    'root_path': '/kaggle/input/2025-basic-p-3-emotion-classification-via-audio',
    'train_path': '/kaggle/input/2025-basic-p-3-emotion-classification-via-audio/train',
    'test_path': '/kaggle/input/2025-basic-p-3-emotion-classification-via-audio/test',
    'submit_path': '/kaggle/input/2025-basic-p-3-emotion-classification-via-audio/sample_submission.csv',
    'batch_size': 32,
    'epochs': 15,
    'lr': 5e-4,
    'seed_val': 42,     # 절대 수정하지 마세요.
    'patience': 8
}

In [3]:
# 랜덤시드 고정하기
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(args['seed_val'])

# 디바이스 선택
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
%pip install torchsummary
from torchsummary import summary as summary_ # 모델 정보를 확인하기 위해 torchsummary 함수 import

# 모델의 형태를 출력하기 위한 함수
def summary_model(model, input_shape=(15,)):
    model = model.to(device)
    summary_(model, input_shape) # (model, (input shape))

Note: you may need to restart the kernel to use updated packages.


# **0. 음성 데이터 재생**

# **1. 데이터**

## **(1) Label Map**
{'강아지' : 0, '고양이' : 1} 등과 같은 형식으로, 머신러닝, 딥러닝 모델들은 feature나 label의 값들이 숫자(정수/실수)인 것만 처리할 수 있기 때문에, 문자열일 경우 숫자형으로 변환하여 처리합니다.

In [5]:
label_map = {"Angry": 0, "Disgust": 1, "Fear": 2, "Happy": 3, "Sad": 4, "Neutral": 5}

이 매핑은 모델 학습 시 라벨을 숫자로 변환하거나, 예측 결과(숫자)를 다시 텍스트로 해석할 때 사용됩니다.


## **(2) Feature Extract**
컴퓨터에 입력할 수 있도록 샘플링과 양자화를 거쳤다고 해도, 이를 바로 분류기에 넣는 것은 높은 성능을 보장할 수 없습니다.<br>
이는 음성 정보 안에 여러 주파수가 섞여 있으며 방대한 정보를 담고 있기 때문입니다.<br>
따라서 데이터 중 음성의 대표적인 성질을 나타낼 수 있는 handcrafted feature를 추출하여 사용하는 것이 필수적입니다.<br>
데이터에서 어떠한 특징을 어떠한 방식으로 추출하는지에 따라 분류기 성능에 큰 영향을 끼칠 수 있기 때문에 feature 추출을 위해서는 신중한 설계가 선행되어야 합니다.<br>

- 참고 1 : https://librosa.org/doc/latest/index.html
- 참고 2 : https://wikidocs.net/192879

#### **Method. MFCC + Pitch + RMS + ZCR**
- 이 방법은 네 가지 feature인 MFCC, Pitch, RMS, ZCR을 시간 프레임 단위로 추출하여 통합합니다.
- 각 단계별 세부 내용은 다음과 같습니다.

(1-1) librosa.load(): 파일 경로에서 22050Hz 샘플링 레이트로 3초 길이의 오디오를 로드합니다 (0.5초 오프셋 적용)

(1-2) librosa.feature.mfcc(): 로드된 time-series에 13개의 MFCC 계수를 추출하여 (time_step, 13) 형태로 변환합니다.

(1-3) librosa.piptrack():
주파수-시간 영역에서 가장 높은 주파수 에너지를 가진 피치 값을 추출하여 (time_step, 1) 형태로 정렬합니다.

(1-4) librosa.feature.rms(): 오디오의 RMS 에너지(Root Mean Square Energy) 를 프레임 단위로 계산하고 (time_step, 1)로 정렬합니다.

(1-5) librosa.feature.zero_crossing_rate(): 오디오의 영교차율(Zero Crossing Rate) 을 프레임 단위로 계산하고 (time_step, 1)로 정렬합니다.

(1-6) MFCC(13차원), Pitch(1차원), RMS(1차원), ZCR(1차원)을 하나의 feature로 반환할 수 있도록 concatenate를 수행하여 (time_step, 16)의 형태를 갖습니다.

In [6]:
def extract_features(file_path, frame_length=2048, hop_length=512, n_mfcc=13):
    """
    주어진 오디오 파일 경로(file_path)에서 음성 감정 인식에 유용한 특징(feature)들을 추출합니다.
    추출하는 특성:
        - MFCC (Mel-Frequency Cepstral Coefficients)
        - Pitch
        - RMS Energy
        - Zero Crossing Rate (ZCR)

    Parameters:
        file_path (str): 오디오 파일(.wav 등)의 경로
        frame_length (int): 한 프레임당 샘플 수 (에너지/ZCR 계산에 사용)
        hop_length (int): 프레임 간 이동 간격 (time resolution 조절)
        n_mfcc (int): 추출할 MFCC 계수 개수

    Returns:
        features (np.ndarray): (time_step, feature_dim) 형태의 feature 행렬
                                → 각 time step마다 16개의 특징이 포함됨
    """

    # (1-1) librosa.load(): 오디오 파일을 로드하여 시계열 신호(y)와 샘플링 레이트(sr)를 반환합니다.
    # - sr: 1초에 22050개의 샘플로 고정
    # - duration: 전체 음성 중 3초만 사용
    # - offset: 시작 시점에서 0.5초를 건너뜁니다.
    # - y: 오디오 신호를 나타내는 1차원 NumPy 배열로, 각 요소는 시간 순서대로 정규화된 진폭 값을 의미합니다. (값 범위: -1.0 ~ 1.0)
    y, sr = librosa.load(file_path, sr=22050, duration=3.0, offset=0.5)

    # (1-2) librosa.feature.mfcc(): MFCC 추출합니다.
    # - n_mfcc: 13개의 MFCC 계수를 사용 → shape: (T, 13)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=frame_length, hop_length=hop_length)
    mfcc = mfcc.T  # (feature_dim, time_step) → (time_step, feature_dim) 형태로 시간축이 앞으로 오도록 전치

    # (1-3) librosa.piptrack(): 시간-주파수 영역에서 피치를 추출합니다.
    # pitches: (주파수 bins, 시간 frames) 형태의 2차원 배열입니다.
    # → 각 시간 프레임별로 여러 주파수 bin의 후보 피치가 존재함
    # pitch: 각 시간 프레임에서 에너지가 가장 큰 주파수 bin의 피치값만 선택하여 1차원 배열로 변환합니다.
    # - 즉, 다차원 주파수 후보 중에서 대표값(최댓값)을 추출해 시계열 형태로 단순화한 것
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    pitch = np.array([pitches[:, t].max() if pitches[:, t].max() > 0 else 0 for t in range(pitches.shape[1])])
    pitch = pitch.reshape(-1, 1)  # 시간 축 기준으로 max 값을 추출했기 때문에 (T, 1) 형태로 reshape 하기
    # shape: (T, 1)

    # (1-4) librosa.feature.rms(): RMS (Root Mean Square) 에너지를 추출합니다.
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)
    rms = rms.T
    # rms shape: (T, 1)

    # (1-5) librosa.feature.zero_crossing_rate(): ZCR (Zero Crossing Rate)를 추출합니다.
    zcr = librosa.feature.zero_crossing_rate(y=y, frame_length=frame_length, hop_length=hop_length)
    zcr = zcr.T
    # zcr shape: (T, 1)

    # (1-6) 모든 특성(feature)들을 하나의 벡터로 통합합니다.
    # - MFCC(13), Pitch(1), RMS(1), ZCR(1) → 총 16차원의 feature
    # 시간 축을 기준으로 연결하려면 축(axis)을 어떻게 설정해야 할까요?
    features = np.concatenate([mfcc, pitch, rms, zcr], axis=1)
    # features shape:n_mfcc + pitch + rms + zcr  (T, 16)
    return features

## **(3) Custom Dataset**
Custom Dataset 클래스는 크게 **__init__(), __len__(), 그리고 __getitem__()** 3개의 함수로 구현해야 합니다.

1.  __init__()
    - Dataset instance를 생성할 때 한번만 실행되는 함수로, 입력 이미지의 디렉토리와 라벨 정보 그리고 transform을 초기화 합니다.

2. __len__()
    - 데이터셋의 샘플 개수를 반환하는 함수 입니다.
    
3. __getitem__()
    - 주어진 인덱스에 해당하는 데이터 샘플을 데이터셋에서 불러오고 반환하는 함수 입니다.

In [7]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, label_map, root_path=None, split="Train"):
        """
        CustomDataset 클래스

        Args:
            dataframe (pd.DataFrame): Id(파일명), Emotions(감정 라벨) 등의 열이 포함된 전체 데이터프레임
            label_map (dict): 감정 라벨(str) → 정수 인덱스(int) 매핑 딕셔너리
            root_path (str): 오디오 파일들이 저장된 루트 디렉토리
            split (str): 'Train' 또는 'Test' 여부
        """
        self.df = dataframe.reset_index(drop=True)
        self.label_map = label_map
        self.root_path = root_path
        self.split = split.upper()

    def __len__(self):
        # (2-1) 전체 샘플 개수 반환
        return len(self.df)

    def __getitem__(self, idx):
        # (2-2) idx번째 오디오 파일 이름 추출
        audio_filename = self.df.loc[idx, 'Id']

        # split에 따라 'TEST' 일때 test 경로 사용하고 그 외에 train 사용
        if self.split == 'TEST':
            audio_path = os.path.join(self.root_path, 'test', audio_filename)
        else:
            audio_path = os.path.join(self.root_path, 'train', audio_filename)

        # (2-3) 오디오에서 feature 추출
        features = extract_features(audio_path)

        # (2-4) numpy 배열을 모델에 입력시키기 위해 torch.Tenosr로 변환
        features = torch.tensor(features, dtype=torch.float32)

        # (2-5 Test셋은 label 없으므로 feature만 반환
        if self.split == 'TEST':
            return features

        # (2-6) Train셋은 라벨 정보를 매핑하여 feature와 label 함께 반환
        label_name = self.df.loc[idx, 'Emotions']        # label: 'haapy'
        label = self.label_map[label_name]               # label: 1

        return features, label

## **(4) collate_fn 함수**
collate_fn은 PyTorch의 DataLoader에서 데이터를 불러올 때, 오디오 길이가 서로 다른 feature들을 동일한 길이로 맞춰주는 사용자 정의 함수입니다. 이 과정을 통해 모델이 고정된 입력 크기를 받을 수 있도록 도와줍니다.

1. pad_sequence()
    - 서로 길이가 다른 시퀀스를 가장 긴 시퀀스에 맞춰 0으로 패딩합니다.

    - batch_first=True: 출력 텐서의 첫 번째 차원이 배치 크기(batch_size)가 되도록 설정합니다.

    - .permute(0, 2, 1): pad_sequence의 결과는 (batch_size, max_seq_len, feature_dim) 형태이지만, 모델 입력을 위해 (batch_size, feature_dim, max_seq_len) 형태로 변환합니다.

2. 레이블 처리
    - 배치에 (feature, label) 쌍으로 되어있으면 zip(*batch)로 feature와 label을 각각 분리합니다.

    - label은 torch.tensor()로 long 타입 변환하여 텐서로 반환합니다.

3. 입력 형태에 따라 처리
    - Train: (feature, label) 형태의 데이터셋 → feature와 label 각각 반환

    - Test: label이 없는 테스트 데이터 등 → feature만 반환

In [8]:
# 모든 오디오 feature 시퀀스를 동일한 길이로 padding 처리하는 collate 함수
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    """
    DataLoader에서 배치 단위로 데이터를 불러올 때,
    시퀀스 길이가 서로 다른 feature들을 동일한 길이로 패딩 시켜주는 함수입니다.

    - Train 데이터의 경우: (feature, label) 튜플 형식
    - Test 데이터의 경우: feature만 존재

    Args:
        batch (list): Dataset으로부터 받은 배치 데이터 (리스트 형태)

    Returns:
        padded_features: shape (B, feature_dim, max_len)
        labels (optional): shape (B,) → Train일 경우에만 존재
    """

    # Train 데이터: (features, label) 쌍
    if isinstance(batch[0], tuple):
        features, labels = zip(*batch)

        # (3-1) 서로 다른 길이의 feature들을 가장 긴 시퀀스에 맞춰 0으로 패딩 → shape: [B, max_len, feature_dim]
        padded_features = pad_sequence(features, batch_first=True)

        # (3-2) permute(): 모델 입력 형태에 맞춰 shape을 [B, feature_dim, max_len]로 변환
        padded_features = padded_features.permute(0, 2, 1)

        # (3-3) 라벨 리스트를 LongTensor로 변환 → shape [B]
        labels = torch.tensor(labels, dtype=torch.long)

        # (3-4) Train 시 feature와 label 함께 반환
        return padded_features, labels

    else:
        # Test 데이터: label 없이 feature만 존재
        # (3-6) 서로 다른 길이의 feature들을 가장 긴 시퀀스에 맞춰 0으로 패딩 → shape: [B, max_len, feature_dim]
        padded_features = pad_sequence(batch, batch_first=True)

        # (3-6) 모델 입력 형태에 맞춰 shape: [B, feature_dim, max_len]로 변환
        padded_features = padded_features.permute(0, 2, 1)

        # (3-7) Test 시 feature만 반환
        return padded_features

## **(5) 데이터셋과 데이터로더**


In [9]:
# (4-1) CSV 로딩
# - 학습(train.csv), 테스트(test.csv) 데이터를 불러옵니다.
full_df = pd.read_csv(os.path.join(args['root_path'], 'train.csv'))
test_df = pd.read_csv(os.path.join(args['root_path'], 'test.csv'))

# (4-2) train과 valid 나누기
# - 전체 데이터 중 10%를 validation set으로 사용
# - random_state: seed 고정으로 매번 동일하게 분할되도록 설정
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(full_df, test_size=0.1, random_state=42, shuffle=True)

In [10]:
# (4-3) 데이터셋 로드
train_dataset = CustomDataset(train_df, label_map, root_path=args['root_path'], split='Train')
valid_dataset = CustomDataset(val_df, label_map, root_path=args['root_path'], split='Train')
test_dataset = CustomDataset(test_df, label_map, root_path=args['root_path'], split='Test')

# (4-4) 데이터로더 정의 collate fn 사용
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=args['batch_size'], 
                                               shuffle=True, collate_fn=collate_fn)
valid_dataloader = torch.utils.data.DataLoader(valid_dataset, batch_size=args['batch_size'], 
                                               shuffle=False, collate_fn=collate_fn)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=args['batch_size'], 
                                              shuffle=False, collate_fn=collate_fn)

In [11]:
# train_loader에서 첫 번째 배치 꺼내기
train_batch = next(iter(train_dataloader))
train_features, train_labels = train_batch
print(f"Train Batch - Feature shape: {train_features.shape}, Label shape: {train_labels.shape}")

# test_loader는 label이 없으므로 unpack 없이 바로 feature만 꺼냄
test_batch = next(iter(test_dataloader))
print(f"Test Batch - Feature shape: {test_batch.shape}")

Train Batch - Feature shape: torch.Size([32, 16, 130]), Label shape: torch.Size([32])
Test Batch - Feature shape: torch.Size([32, 16, 117])


# **2. 모델**
모델을 직접 설계하여 프로젝트를 수행하세요.

> ```
> ----------------------------------------------------------------  
>         Layer (type)               Output Shape         Param #  
> =================================================================  
>             Conv1d-1              [-1, 64, 130]           5,184  
>        BatchNorm1d-2              [-1, 64, 130]             128  
>               ReLU-3              [-1, 64, 130]               0  
>             Conv1d-4              [-1, 64, 130]          12,352  
>        BatchNorm1d-5              [-1, 64, 130]             128  
>               ReLU-6              [-1, 64, 130]               0  
>            Dropout-7              [-1, 64, 130]               0  
>          MaxPool1d-8               [-1, 64, 65]               0  
>             Conv1d-9              [-1, 128, 65]          24,704  
>       BatchNorm1d-10              [-1, 128, 65]             256  
>              ReLU-11              [-1, 128, 65]               0  
>            Conv1d-12              [-1, 128, 65]          49,280  
>       BatchNorm1d-13              [-1, 128, 65]             256  
>              ReLU-14              [-1, 128, 65]               0  
>            Conv1d-15              [-1, 128, 65]          49,280  
>       BatchNorm1d-16              [-1, 128, 65]             256  
>              ReLU-17              [-1, 128, 65]               0  
>           Dropout-18              [-1, 128, 65]               0  
>         MaxPool1d-19              [-1, 128, 32]               0  
>            Conv1d-20              [-1, 256, 32]          98,560  
>       BatchNorm1d-21              [-1, 256, 32]             512  
>              ReLU-22              [-1, 256, 32]               0  
>            Conv1d-23              [-1, 256, 32]         196,864  
>       BatchNorm1d-24              [-1, 256, 32]             512  
>              ReLU-25              [-1, 256, 32]               0  
>           Dropout-26              [-1, 256, 32]               0  
>  AdaptiveAvgPool1d-27               [-1, 256, 1]               0  
>           Flatten-28                  [-1, 256]               0  
>            Linear-29                  [-1, 128]          32,896  
>              ReLU-30                  [-1, 128]               0  
>           Dropout-31                  [-1, 128]               0  
>            Linear-32                    [-1, 6]             774  
> =================================================================  
> Total params: 471,942  
> Trainable params: 471,942  
> Non-trainable params: 0  
> ----------------------------------------------------------------  
> Input size (MB): 0.01  
> Forward/backward pass size (MB): 1.59  
> Params size (MB): 1.80  
> Estimated Total Size (MB): 3.39  
> ----------------------------------------------------------------  
> ```


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1) 시간축 주의 풀링
class AttentionPooling1D(nn.Module):
    def __init__(self, in_channels, hidden=128):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv1d(in_channels, hidden, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(hidden, 1, kernel_size=1)
        )
    def forward(self, x):          # x: (B, C, T)
        a = self.attn(x)           # (B, 1, T)
        w = F.softmax(a, dim=-1)   # 시간축 가중치
        return (x * w).sum(dim=-1) # (B, C)

# 2) 채널 주의(SE)
class SE1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Conv1d(channels, channels // reduction, 1)
        self.fc2 = nn.Conv1d(channels // reduction, channels, 1)
    def forward(self, x):                       # x: (B, C, T)
        s = x.mean(dim=-1, keepdim=True)        # GAP
        z = F.relu(self.fc1(s), inplace=True)
        z = torch.sigmoid(self.fc2(z))
        return x * z

# 3) (옵션) 코사인 분류기
class CosineClassifier(nn.Module):
    def __init__(self, in_dim, num_classes, s=30.0, learn_s=False):
        super().__init__()
        self.W = nn.Parameter(torch.empty(num_classes, in_dim))
        nn.init.xavier_uniform_(self.W)
        self.s = nn.Parameter(torch.tensor(float(s))) if learn_s else float(s)
    def forward(self, x):  # x: (B, D)
        x = F.normalize(x, dim=-1)
        W = F.normalize(self.W, dim=-1)
        return (x @ W.t()) * (self.s if isinstance(self.s, float) else self.s)

# 4) 개선된 모델
class Model(nn.Module):
    def __init__(self, use_cosine=False, p_feat=0.2, p_cls=0.4, n_classes=6):
        super(Model, self).__init__()
        self.use_cosine = use_cosine

        self.features = nn.Sequential(
            # in: (B, 16, T)
            nn.Conv1d(16, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=p_feat),
            SE1D(64),                              # ← 추가
            nn.MaxPool1d(kernel_size=2, stride=2), # T -> T/2

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=p_feat),
            SE1D(128),                             # ← 추가
            nn.MaxPool1d(kernel_size=2, stride=2), # T/2 -> T/4

            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=p_feat),
            SE1D(256),                             # ← 추가
            # 기존 AdaptiveAvgPool1d(1) 제거, 아래 AttentionPooling으로 대체
        )

        self.pool = AttentionPooling1D(256, hidden=128)   # ← 교체된 풀링

        self.proj = nn.Sequential(                        # 분류기 입력 전 투영
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=p_cls)
        )

        if self.use_cosine:
            self.classifier = CosineClassifier(256, n_classes, s=30.0, learn_s=False)
        else:
            self.classifier = nn.Linear(256, n_classes)

    def forward(self, x):
        h = self.features(x)   # (B, 256, T')
        z = self.pool(h)       # (B, 256)
        z = self.proj(z)       # (B, 256)
        logits = self.classifier(z)  # (B, n_classes)
        return logits

In [13]:
# 모델을 장치로 이동
model = Model().to(device)

# 모델 요약 정보 출력
summary_(model, input_size=train_features.shape[1:])

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv1d-1              [-1, 64, 130]           5,184
       BatchNorm1d-2              [-1, 64, 130]             128
              ReLU-3              [-1, 64, 130]               0
            Conv1d-4              [-1, 64, 130]          12,352
       BatchNorm1d-5              [-1, 64, 130]             128
              ReLU-6              [-1, 64, 130]               0
           Dropout-7              [-1, 64, 130]               0
            Conv1d-8                 [-1, 4, 1]             260
            Conv1d-9                [-1, 64, 1]             320
             SE1D-10              [-1, 64, 130]               0
        MaxPool1d-11               [-1, 64, 65]               0
           Conv1d-12              [-1, 128, 65]          24,704
      BatchNorm1d-13              [-1, 128, 65]             256
             ReLU-14              [-1, 

# **3. 학습**

In [14]:
def train(train_dataloader, valid_dataloader, model,  device, args):
    """
    모델을 훈련시키는 함수입니다.

    Args:
        train_dataloader (DataLoader): 훈련 데이터로더
        valid_dataloader (DataLoader): 검증 데이터로더
        model (nn.Module): 학습할 모델
        optimizer: 파라미터 업데이트를 위한 옵티마이저
        loss_fn: 손실 함수
        device (torch.device): 연산을 수행할 장치 (CPU 또는 GPU)
        args (dict): 훈련 설정 및 하이퍼파라미터 딕셔너리

    Returns:
        None
    """
    # (6-1) Adam 옵티마이저와 교차 엔트로피 손실 함수 정의
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])

    # 모델을 장치로 이동
    model.to(device)

    best_val_acc = 0.0 # 최고 검증 정확도 저장용 변수
    patience = args['patience'] # 조기 종료를 위한 값
    patience_counter = 0 # 현재 patience 카운터

    # 지정된 에포크 수만큼 훈련 반복
    for epoch in range(args['epochs']):
        print(f"Epoch {epoch + 1}/{args['epochs']}")

        # ===== Training =====
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0

        # 훈련 데이터로더에서 데이터를 반복적으로 가져옴
        for data, label in tqdm(train_dataloader, desc="Training"):
            # 데이터를 부동 소수점 형식으로 변환하고 장치에 올림
            data = data.float().to(device)

            # 라벨도 장치에 올림
            label = label.to(device)

            # (6-2) 모델을 사용하여 예측 수행
            pred = model(data)

            # (6-3) 예측값과 실제 라벨을 사용하여 손실 계산 손실 함수에 따라 입력 파라미터 수정
            loss = loss_fn(pred, label)

            # (6-4) 옵티마이저의 기울기를 초기화
            optimizer.zero_grad()

            # (6-5) 역전파를 통해 기울기 계산
            loss.backward()

            # (6-6) 옵티마이저를 사용하여 모델의 파라미터 업데이트
            optimizer.step()

            # 총 손실 계산
            train_loss += loss.item()

            # 예측값에서 가장 높은 값의 인덱스를 선택 해당 인덱스값이 모델이 추출한 예측 라벨 값
            pred = pred.argmax(dim=1)

            # 정확히 예측한 개수 누적
            correct += (pred == label).sum().item()
            total += label.size(0)

        # 평균 훈련 손실 계산
        avg_train_loss = train_loss / len(train_dataloader)

        # 훈련 정확도 계산 Accuracy를 기준으로 함
        train_acc = correct / total

        # ===== Validation =====

        # 모델을 평가 모드로 설정
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        # (6-7) 평가 시에는 그래디언트 계산 비활성화
        with torch.no_grad():
            for data, label in tqdm(valid_dataloader, desc="Validation"):
                data = data.float().to(device)
                label = label.to(device)

                # (6-8) 6-2와 같은 방식으로 데이터만 다르게 모델의 예측 수행
                pred = model(data)

                # (6-9) 6-3와 같은 방식으로 validation 데이터 손실값 계산
                loss = loss_fn(pred, label)

                val_loss += loss.item()
                pred = pred.argmax(dim=1)
                val_correct += (pred == label).sum().item()
                val_total += label.size(0)

        # 평균 검증 손실 계산
        avg_val_loss = val_loss / len(valid_dataloader)

        # 검증 정확도 계산
        val_acc = val_correct / val_total

        # 현재 에포크의 손실과 정확도 출력
        print(f"Train Loss : {avg_train_loss:.4f}, Train Accuracy : {train_acc:.4f}")
        print(f"Val   Loss : {avg_val_loss:.4f}, Val   Accuracy : {val_acc:.4f}\n")

        # 모델과 옵티마이저 상태 저장
        os.makedirs("results", exist_ok=True)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_val_loss,
        }, f'./results/model_state_dict_epoch_{epoch+1}.pth')


        # best model 따로 저장
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_val_loss
                }, f'./results/best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience: # patience 초과하면 조기 종료
                print("Early stopping.")
                break

if __name__ == "__main__":
    # Train 함수 호출
    train(train_dataloader, valid_dataloader, model, device, args)

Epoch 1/15


Validation: 100%|██████████| 21/21 [00:20<00:00,  1.03it/s]


Train Loss : 1.5203, Train Accuracy : 0.3750
Val   Loss : 1.5100, Val   Accuracy : 0.3403

Epoch 2/15


Validation: 100%|██████████| 21/21 [00:13<00:00,  1.53it/s]


Train Loss : 1.4190, Train Accuracy : 0.4316
Val   Loss : 1.4748, Val   Accuracy : 0.4090

Epoch 3/15


...

Train Loss : 1.0976, Train Accuracy : 0.5814
Val   Loss : 1.1530, Val   Accuracy : 0.5567

Epoch 14/15


Validation: 100%|██████████| 21/21 [00:14<00:00,  1.46it/s]


Train Loss : 1.0657, Train Accuracy : 0.5986
Val   Loss : 1.2299, Val   Accuracy : 0.5090

Epoch 15/15


Validation: 100%|██████████| 21/21 [00:15<00:00,  1.40it/s]

Train Loss : 1.0547, Train Accuracy : 0.5983
Val   Loss : 1.2082, Val   Accuracy : 0.5269



# **4. 평가**

In [15]:
def test(test_dataloader, model, device):
    """
    모델의 예측을 수행하는 함수입니다.

    Args:
        test_dataloader (DataLoader): 테스트 데이터가 포함된 데이터로더
        model (nn.Module): 예측에 사용할 신경망 모델
        device (torch.device): 모델과 데이터를 올릴 장치 (CPU 또는 GPU)

    Returns:
        preds (list): 예측된 클래스의 정수 인덱스 리스트
    """
    # 모델을 평가 모드로 전환
    model.eval()

    # 예측 결과를 저장할 리스트 초기화
    preds = []

    # 기울기 계산을 비활성화하여 예측 수행
    with torch.no_grad():

        # 데이터로더에서 데이터를 반복적으로 가져옴
        for data in tqdm(test_dataloader, desc="Testing"):

            # 데이터를 부동 소수점 형식으로 변환하고 장치에 올림
            data = data.float().to(device)

            # (7-1) 모델을 사용하여 예측 수행
            pred = model(data)

            # (7-2) argmax를 이용하여 예측 결과에서 가장 높은 값의 인덱스를 선택
            pred_classes = pred.argmax(dim=1)

            # (7-3) 예측 결과를 리스트에 추가
            preds.extend(pred_classes.cpu().numpy())

    return preds

def int_to_label(label_map, predictions_int):
    """
    정수 형태의 예측값 리스트를 클래스 라벨로 변환하는 함수입니다.

    Args:
        label_map (dict): 클래스 라벨과 정수 인덱스 매핑이 저장된 딕셔너리
        predictions_int (list): 정수 형태의 예측값 리스트

    Returns:
        predicted_labels (list): 문자열 형태의 예측 클래스 라벨 리스트
    """
    # 정수 예측값을 클래스 라벨로 변환하여 리스트에 저장
    reverse_label_map = {v: k for k, v in label_map.items()}
    predicted_labels = [reverse_label_map[pred] for pred in predictions_int]
    return predicted_labels

if __name__ == "__main__":
    # 예측값을 얻기 위해 test 함수 호출
    preds = test(test_dataloader, model, device)

    # 정수 예측값을 클래스 라벨로 변환
    preds = int_to_label(label_map, preds)

Testing: 100%|██████████| 24/24 [00:23<00:00,  1.02it/s]


In [16]:
submit = pd.read_csv(args["submit_path"])
submit['Emotions'] = preds
submit.to_csv('//kaggle/working/results/submission_p3.csv', index=False)

In [17]:
submit

,Id,Emotions
0,Test_0.wav,Disgust
1,Test_1.wav,Neutral
2,Test_2.wav,Sad
3,Test_3.wav,Neutral
4,Test_4.wav,Neutral
...,...,...
740,Test_740.wav,Happy
741,Test_741.wav,Angry
742,Test_742.wav,Disgust
743,Test_743.wav,Happy
